# Fine-tune and Evaluate OPT-2.7B

This notebook uses the same controlled experimental pipeline as the other OPT model notebook.

- Same train, validation and test splits.
- Same external `prompt_templates.json`.
- Same seven-aspect target.
- Same 2,048-token context handling.
- Same 4-bit NF4 QLoRA configuration.
- Same LoRA configuration using all linear layers.
- Same optimizer, scheduler, warmup, gradient clipping, batch size and learning rate.
- Same maximum epochs, early stopping and best-checkpoint selection.
- Same deterministic single-pass generation.
- Same evaluation metrics.

The intended experimental variable is the OPT base-model size.

## Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

In [ ]:
# %pip install -q "transformers==4.57.6" "peft==0.20.0" "accelerate==1.14.0" "bitsandbytes==0.50.0" rouge-score textstat

In [ ]:
# %pip install -r requirements.txt

In [ ]:
import re
import json
import time
import random
import platform
from functools import partial
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

import transformers
import peft
import accelerate
import bitsandbytes as bnb

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    TrainerCallback,                     # ADDED: quality-aware checkpoint evaluation.
    set_seed,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    get_peft_model_state_dict,           # ADDED: keep the best quality adapter in memory.
    set_peft_model_state_dict,           # ADDED: restore the best quality adapter.
)
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer  # ADDED: source-only keyword guidance.
import textstat

PROCESSED_DIR = "data/processed"
TRAIN_PATH = os.path.join(PROCESSED_DIR, "train.jsonl")
VAL_PATH = os.path.join(PROCESSED_DIR, "validation.jsonl")
TEST_PATH = os.path.join(PROCESSED_DIR, "test.jsonl")
TEMPLATES_PATH = os.path.join(PROCESSED_DIR, "prompt_templates.json")

MODEL_NAME = "facebook/opt-2.7b"

# CHANGED: use separate output names so this optimized experiment does not overwrite the baseline.
FINETUNED_MODEL_DIR = "models/opt-2.7b-news-model"
PHASE1_MODEL_DIR = "models/opt-2.7b-summary-model"
RESULTS_DIR = "results/opt-2_7b"
RESULT_PREFIX = "opt2_7b"

SEED = 42
MAX_CONTEXT = 2048
MAX_TARGET_TOKENS = 384
MAX_NEW_TOKENS = 384

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# ------------------------------------------------------------------
# CHANGED: Optimization 1 uses Stage-1 files prepared separately by
# download_dataset_stage1.ipynb. No download/streaming happens here.
# ------------------------------------------------------------------
STAGE1_TRAIN_PATH = os.path.join(PROCESSED_DIR, 'stage1_train.jsonl')
STAGE1_VAL_PATH = os.path.join(PROCESSED_DIR, 'stage1_validation.jsonl')

# Stage 1 teaches broad news summarization first.
PHASE1_NUM_TRAIN_EPOCHS = 5
PHASE1_LEARNING_RATE = 5e-5
PHASE1_EVAL_STEPS = 100
PHASE1_SAVE_STEPS = 100

# Stage 2 adapts the same LoRA weights to the project's Summary + 8 Aspects task.
# Lower LR reduces the chance of overwriting the summary ability learned in Stage 1.
PHASE2_NUM_TRAIN_EPOCHS = 10
PHASE2_LEARNING_RATE = 2e-5
PHASE2_EVAL_STEPS = 25
PHASE2_SAVE_STEPS = 25

# ADDED: Optimization 3 - validation generation is used to select the
# best adapter, not validation loss alone.
QUALITY_EVAL_SAMPLES = 20
QUALITY_EVAL_MAX_NEW_TOKENS = 256
QUALITY_EARLY_STOPPING_PATIENCE = 4

# ADDED: Optimization 4 - deterministic source-keyword guidance.
# Keywords are extracted ONLY from article text, never from reference summaries
# or generated aspects, so they cannot introduce new factual claims.
USE_KEYWORD_GUIDANCE = True
KEYWORD_TOP_K = 8
KEYWORD_MAX_FEATURES = 30000

TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 0.3

os.makedirs(FINETUNED_MODEL_DIR, exist_ok=True)
os.makedirs(PHASE1_MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-capable GPU is required for QLoRA training.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GPU_CAPABILITY = torch.cuda.get_device_capability(0)
USE_BF16 = GPU_CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)
print("Compute dtype:", COMPUTE_DTYPE)

## Load data and prompt template

In [ ]:
REQUIRED_TOP_LEVEL_FIELDS = {"article_id", "article", "summary", "aspects"}
ASPECT_KEYS = ('event', 'people', 'organization', 'date_time', 'location', 'causes', 'effects', 'countermeasures')

def load_jsonl(path: str) -> pd.DataFrame:
    """Load the original project dataset. Existing schema is unchanged."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find '{path}'.")

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            record = json.loads(line)
            missing_top = REQUIRED_TOP_LEVEL_FIELDS - set(record)
            if missing_top:
                raise ValueError(
                    f"{path}, line {line_number}: "
                    f"missing fields {sorted(missing_top)}"
                )

            if not isinstance(record["aspects"], dict):
                raise ValueError(
                    f"{path}, line {line_number}: 'aspects' must be an object."
                )

            missing_aspects = [key for key in ASPECT_KEYS if key not in record['aspects']]
            if missing_aspects:
                raise ValueError(
                    f"{path}, line {line_number}: "
                    f"missing aspect fields {missing_aspects}"
                )

            records.append(record)

    if not records:
        raise ValueError(f"No records found in '{path}'.")

    return pd.DataFrame(records)

# Original 500-row project splits. The test split is NEVER used for training.
train_df = load_jsonl(TRAIN_PATH)
val_df = load_jsonl(VAL_PATH)
test_df = load_jsonl(TEST_PATH)

if not os.path.exists(TEMPLATES_PATH):
    raise FileNotFoundError(f"Could not find '{TEMPLATES_PATH}'.")

with open(TEMPLATES_PATH, "r", encoding="utf-8") as f:
    prompt_templates = json.load(f)

PROMPT_TEMPLATE = prompt_templates["prompt_template"]

if PROMPT_TEMPLATE.count("{article}") != 1:
    raise ValueError(
        "prompt_template must contain {article} exactly once."
    )

# ADDED: protect the final 8-aspect output format.
REQUIRED_PROMPT_LABELS = (
    "Summary:",
    "Aspects:",
    "Event:",
    "People:",
    "Organization:",
    "Date & Time:",
    "Location:",
    "Causes:",
    "Effects:",
    "Countermeasures:",
)

missing_prompt_labels = [label for label in REQUIRED_PROMPT_LABELS if label not in PROMPT_TEMPLATE]

if missing_prompt_labels:
    raise ValueError(
        "The prompt template is missing labels: "
        f"{missing_prompt_labels}"
    )

# ------------------------------------------------------------------
# CHANGED: Stage 1 is loaded only from local JSONL files created by
# download_dataset_stage1.ipynb.
# ------------------------------------------------------------------
STAGE1_REQUIRED_FIELDS = {'article_id', 'article', 'summary'}

def load_stage1_jsonl(
    path: str,
) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Could not find '{path}'. "
            "Run download_dataset_stage1.ipynb first."
        )

    records = []

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:
        for line_number, line in enumerate(
            f,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            missing_fields = STAGE1_REQUIRED_FIELDS - set(record)

            if missing_fields:
                raise ValueError(
                    f"{path}, line {line_number}: "
                    f"missing fields "
                    f"{sorted(missing_fields)}"
                )

            article = str(record['article']).strip()

            summary = str(record['summary']).strip()

            if not article or not summary:
                raise ValueError(
                    f"{path}, line {line_number}: "
                    "article and summary must be non-empty."
                )

            records.append(
                {
                    "article_id": str(
                        record["article_id"]
                    ).strip(),
                    "article": article,
                    "summary": summary,
                    "article_word_count": (
                        record.get(
                            "article_word_count"
                        )
                    ),
                }
            )

    if not records:
        raise ValueError(
            f"No Stage-1 records found in '{path}'."
        )

    return pd.DataFrame(records)


stage1_source_train_df = load_stage1_jsonl(STAGE1_TRAIN_PATH)

stage1_source_val_df = load_stage1_jsonl(STAGE1_VAL_PATH)

# ADDED: verify Stage-1 train/validation files do not overlap.
stage1_train_ids = set(stage1_source_train_df['article_id'].astype(str))

stage1_val_ids = set(stage1_source_val_df['article_id'].astype(str))

stage1_overlap = stage1_train_ids & stage1_val_ids

if stage1_overlap:
    raise ValueError(
        "Stage-1 training and validation files overlap. "
        f"Example overlapping IDs: "
        f"{list(stage1_overlap)[:5]}"
    )

print('Project training:', len(train_df))
print('Project validation:', len(val_df))
print('Project test:', len(test_df))
print('Stage-1 summary training:', len(stage1_source_train_df))
print('Stage-1 summary validation:', len(stage1_source_val_df))
print('Stage-1 train/validation overlap:', len(stage1_overlap))
print("\nOriginal prompt template:\n")
print(PROMPT_TEMPLATE)

## Build target and fit articles to context

In [ ]:
# ------------------------------------------------------------------
# ADDED: Optimization 2 - stronger summary instruction.
# It is short enough to preserve the 2048-token context budget.
# ------------------------------------------------------------------
SUMMARY_GUIDANCE = (
    "For the Summary, state the central news event first. "
    "Include the most important outcome and essential facts before "
    "supporting or background details. "
    "Do not omit facts needed to understand what happened."
)

# ADDED: Stage 1 has a summary-only prompt because the external corpus
# does not contain the project's AI-generated aspect labels.
SUMMARY_ONLY_PROMPT_TEMPLATE = """Simplify the news article using only facts stated in the article.
Use simpler vocabulary, grammar, and sentence structures while preserving the meaning.
Do not guess information.

Output exactly this format:

Summary:
<2 to 4 concise simplified sentences>

{summary_guidance}

{keyword_block}Article:
{article}

Response:
"""

# ADDED: Keyword extractor is fitted ONLY on training article text.
# No validation/test/reference-summary text is used to fit it.
keyword_fit_articles = pd.concat(
    [
        stage1_source_train_df["article"],
        train_df["article"],
    ],
    ignore_index=True,
).astype(str).tolist()

keyword_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=KEYWORD_MAX_FEATURES,
    min_df=2,
    lowercase=True,
)

keyword_vectorizer.fit(keyword_fit_articles)
KEYWORD_FEATURE_NAMES = np.asarray(keyword_vectorizer.get_feature_names_out())

def extract_source_keywords(
    article: str,
    top_k: int = KEYWORD_TOP_K,
) -> list:
    """
    ADDED: Extract deterministic TF-IDF terms directly from the source.
    These are hints, not generated facts or aspect predictions.
    """
    article = str(article).strip()

    if not USE_KEYWORD_GUIDANCE or not article:
        return []

    vector = keyword_vectorizer.transform([article])

    if vector.nnz == 0:
        return []

    row = vector.tocoo()
    pairs = sorted(zip(row.col, row.data), key=lambda item: item[1], reverse=True)

    selected = []
    seen = set()

    for feature_index, _ in pairs:
        term = str(KEYWORD_FEATURE_NAMES[feature_index]).strip()

        if term and term not in seen:
            selected.append(term)
            seen.add(term)

        if len(selected) >= top_k:
            break

    return selected

def build_keyword_block(article: str) -> str:
    keywords = extract_source_keywords(article)

    if not keywords:
        return ""

    return 'Important source terms: ' + '; '.join(keywords) + '\n\n'

# ADDED: Keep every original final output label and order. Only insert
# stronger summary guidance and optional source-keyword hints.
def build_full_prompt_template(template: str) -> str:
    marker = "Article:"

    if marker not in template:
        raise ValueError(
            "Prompt template must contain an 'Article:' section."
        )

    inserted = SUMMARY_GUIDANCE + '\n\n{keyword_block}' + marker

    return template.replace(marker, inserted, 1)

FULL_PROMPT_TEMPLATE = build_full_prompt_template(PROMPT_TEMPLATE)

def build_summary_prompt(article: str) -> str:
    """ADDED: Stage-1 Article -> Summary prompt."""
    article = str(article).strip()

    return SUMMARY_ONLY_PROMPT_TEMPLATE.format(
        summary_guidance=SUMMARY_GUIDANCE,
        keyword_block=build_keyword_block(article),
        article=article,
    )

def build_prompt(article: str) -> str:
    """
    CHANGED: Final Stage-2/test prompt keeps the existing labels and
    adds only the stronger summary instruction + source keywords.
    """
    article = str(article).strip()

    return FULL_PROMPT_TEMPLATE.format(keyword_block=build_keyword_block(article), article=article)

def build_summary_target(summary: str) -> str:
    """ADDED: Stage-1 target."""
    return f"Summary:\n{str(summary).strip()}"

def build_training_target(summary: str, aspects: dict) -> str:
    """Original Stage-2 target labels and order are preserved."""
    return (
        f"Summary:\n{str(summary).strip()}\n\n"
        "Aspects:\n"
        f"Event: {str(aspects['event']).strip()}\n"
        f"People: {str(aspects['people']).strip()}\n"
        f"Organization: {str(aspects['organization']).strip()}\n"
        f"Date & Time: {str(aspects['date_time']).strip()}\n"
        f"Location: {str(aspects['location']).strip()}\n"
        f"Causes: {str(aspects['causes']).strip()}\n"
        f"Effects: {str(aspects['effects']).strip()}\n"
        f"Countermeasures: {str(aspects['countermeasures']).strip()}"
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
tokenizer.model_max_length = MAX_CONTEXT

def token_length(
    text: str,
    add_special_tokens: bool = True,
) -> int:
    return len(tokenizer(str(text), add_special_tokens=add_special_tokens, truncation=False)['input_ids'])

def fit_prompt_to_context(
    article: str,
    reserved_tokens: int,
    prompt_builder=build_prompt,
):
    """
    CHANGED: accepts either the Stage-1 or Stage-2 prompt builder.
    Only article text is truncated; instructions/target remain intact.
    """
    input_limit = MAX_CONTEXT - reserved_tokens

    if input_limit <= 0:
        raise ValueError(
            "reserved_tokens must be smaller than MAX_CONTEXT."
        )

    article = str(article).strip()

    article_ids = tokenizer(article, add_special_tokens=False, truncation=False)['input_ids']

    empty_prompt_tokens = token_length(prompt_builder(''), add_special_tokens=True)

    article_budget = max(0, input_limit - empty_prompt_tokens - 8)

    used_ids = article_ids[:article_budget]

    while True:
        used_article = tokenizer.decode(used_ids, skip_special_tokens=True).strip()

        prompt = prompt_builder(used_article)

        prompt_tokens = token_length(prompt, add_special_tokens=True)

        if prompt_tokens <= input_limit:
            break

        if not used_ids:
            raise ValueError(
                "Prompt instructions alone exceed "
                "the available input context."
            )

        excess = prompt_tokens - input_limit
        used_ids = used_ids[:max(0, len(used_ids) - excess - 4)]

    return (prompt, prompt_tokens, len(used_ids) < len(article_ids))

def prepare_training_split(
    frame: pd.DataFrame,
    split_name: str,
    stage: str,
) -> pd.DataFrame:
    """
    ADDED curriculum:
      stage='summary' -> Article -> Summary
      stage='full'    -> Article -> Summary + 8 Aspects
    """
    if stage not in {"summary", "full"}:
        raise ValueError(
            "stage must be 'summary' or 'full'."
        )

    rows = []

    for _, row in frame.iterrows():
        if stage == "summary":
            target = build_summary_target(row['summary'])
            prompt_builder = build_summary_prompt
        else:
            target = build_training_target(row['summary'], row['aspects'])
            prompt_builder = build_prompt

        target_with_eos = target.strip() + tokenizer.eos_token

        target_tokens = token_length(target_with_eos, add_special_tokens=False)

        if target_tokens > MAX_TARGET_TOKENS:
            raise ValueError(
                f"{split_name} article {row['article_id']} target "
                f"has {target_tokens} tokens; increase "
                "MAX_TARGET_TOKENS so the complete target is preserved."
            )

        (
            prompt,
            prompt_tokens,
            article_was_truncated,
        ) = fit_prompt_to_context(
            row["article"],
            target_tokens,
            prompt_builder=prompt_builder,
        )

        prepared = row.to_dict()
        prepared["prompt"] = prompt
        prepared["training_target"] = target
        prepared["prompt_token_length"] = prompt_tokens
        prepared["target_token_length"] = target_tokens
        prepared['total_training_tokens'] = prompt_tokens + target_tokens
        prepared['article_was_truncated'] = article_was_truncated

        rows.append(prepared)

    result = pd.DataFrame(rows)

    print(f"{split_name}: {len(result)} records, truncated articles: {int(result['article_was_truncated'].sum())}")

    print(
        f"{split_name} total tokens: "
        f"median={result['total_training_tokens'].median():.0f}, "
        f"max={result['total_training_tokens'].max():.0f}"
    )

    return result

# CHANGED: Stage 1 = local article-summary files prepared separately.
phase1_train_df = prepare_training_split(stage1_source_train_df, 'Stage 1 summary training', 'summary')

phase1_val_df = prepare_training_split(stage1_source_val_df, 'Stage 1 summary validation', 'summary')

# CHANGED: Stage 2 = only the project's original training/validation
# rows with Summary + all 8 aspect labels.
phase2_train_df = prepare_training_split(train_df, 'Stage 2 project training', 'full')

phase2_val_df = prepare_training_split(val_df, 'Stage 2 project validation', 'full')

# Preserve downstream variable names for the original final evaluation flow.
train_df = phase2_train_df
val_df = phase2_val_df

test_df = test_df.copy()

test_df['training_target'] = test_df.apply(lambda row: build_training_target(row['summary'], row['aspects']), axis=1)

# ------------------------------------------------------------------
# ADDED: self-contained SARI-style metric for validation checkpoint
# comparison. It measures KEEP / DELETE / ADD operations.
# The final notebook also reports this score on the test set.
# ------------------------------------------------------------------
def _ngram_set(tokens, n):
    return {tuple(tokens[i:i + n]) for i in range(max(0, len(tokens) - n + 1))}

def _safe_f1(precision, recall):
    if precision + recall == 0:
        return 0.0

    return 2.0 * precision * recall / (precision + recall)

def compute_sari_score(
    source: str,
    prediction: str,
    reference: str,
) -> float:
    source_tokens = str(source).lower().split()
    pred_tokens = str(prediction).lower().split()
    ref_tokens = str(reference).lower().split()

    n_scores = []

    for n in range(1, 5):
        source_ngrams = _ngram_set(source_tokens, n)
        pred_ngrams = _ngram_set(pred_tokens, n)
        ref_ngrams = _ngram_set(ref_tokens, n)

        # ADD
        pred_add = pred_ngrams - source_ngrams
        ref_add = ref_ngrams - source_ngrams

        add_correct = len(pred_add & ref_add)
        add_precision = add_correct / len(pred_add) if pred_add else 1.0 if not ref_add else 0.0
        add_recall = add_correct / len(ref_add) if ref_add else 1.0
        add_f1 = _safe_f1(add_precision, add_recall)

        # KEEP
        pred_keep = pred_ngrams & source_ngrams
        ref_keep = ref_ngrams & source_ngrams

        keep_correct = len(pred_keep & ref_keep)
        keep_precision = keep_correct / len(pred_keep) if pred_keep else 1.0 if not ref_keep else 0.0
        keep_recall = keep_correct / len(ref_keep) if ref_keep else 1.0
        keep_f1 = _safe_f1(keep_precision, keep_recall)

        # DELETE
        pred_delete = source_ngrams - pred_ngrams
        ref_delete = source_ngrams - ref_ngrams

        delete_correct = len(pred_delete & ref_delete)
        delete_precision = delete_correct / len(pred_delete) if pred_delete else 1.0 if not ref_delete else 0.0

        n_scores.append((add_f1 + keep_f1 + delete_precision) / 3.0)

    return float(np.mean(n_scores) * 100.0)

QUALITY_ROUGE_SCORER = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

QUALITY_ASPECT_LABEL_MAP = {
    "Event": "event",
    "People": "people",
    "Organization": "organization",
    "Date & Time": "date_time",
    "Location": "location",
    "Causes": "causes",
    "Effects": "effects",
    "Countermeasures": "countermeasures",
}

def quality_extract_summary(
    raw_text: str,
    stage: str,
) -> str:
    text = str(raw_text)

    if stage == "summary":
        match = re.search('(?is)^\\s*Summary\\s*:\\s*(.*)\\Z', text)
    else:
        match = re.search('(?is)^\\s*Summary\\s*:\\s*(.*?)(?=\\n\\s*Aspects\\s*:|\\Z)', text)

    return re.sub('\\s+', ' ', match.group(1)).strip() if match else ''

def quality_extract_aspects(
    raw_text: str,
) -> dict:
    text = str(raw_text)
    extracted = {}

    for label, field_name in QUALITY_ASPECT_LABEL_MAP.items():
        match = re.search(f'(?im)^\\s*{re.escape(label)}\\s*:\\s*(.*?)\\s*$', text)
        extracted[field_name] = match.group(1).strip() if match else ''

    return extracted

def quality_has_complete_structure(
    raw_text: str,
) -> bool:
    labels = ['Summary', 'Aspects', *QUALITY_ASPECT_LABEL_MAP.keys()]

    positions = []

    for label in labels:
        found = list(re.finditer(f'(?im)^\\s*{re.escape(label)}\\s*:', str(raw_text)))

        if len(found) != 1:
            return False

        positions.append(found[0].start())

    return positions == sorted(positions)

def quality_rouge_l(
    reference: str,
    prediction: str,
) -> float:
    return QUALITY_ROUGE_SCORER.score(str(reference), str(prediction))['rougeL'].fmeasure

def quality_aspect_macro_rouge_l(
    reference_aspects: dict,
    generated_aspects: dict,
) -> float:
    return float(np.mean([quality_rouge_l(reference_aspects[key], generated_aspects.get(key, '')) for key in ASPECT_KEYS]))

print("\nStage 1 example prompt:\n")
print(phase1_train_df.loc[0, 'prompt'][:1200])

print("\n--- STAGE 1 TARGET ---\n")
print(phase1_train_df.loc[0, 'training_target'])

print("\nStage 2 example prompt:\n")
print(phase2_train_df.loc[0, 'prompt'][:1200])

print("\n--- STAGE 2 TARGET ---\n")
print(phase2_train_df.loc[0, 'training_target'])

## Create training datasets

In [ ]:
class NewsSimplificationDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        tokenizer,
    ):
        self.prompts = frame["prompt"].tolist()
        self.targets = frame["training_target"].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, index):
        prompt = str(self.prompts[index])

        target = str(self.targets[index]).strip() + self.tokenizer.eos_token

        prompt_ids = self.tokenizer(prompt, add_special_tokens=True, truncation=False)['input_ids']

        target_ids = self.tokenizer(target, add_special_tokens=False, truncation=False)['input_ids']

        if (
            len(prompt_ids)
            + len(target_ids)
            > MAX_CONTEXT
        ):
            raise ValueError(
                f"Training pair {index} exceeds "
                f"{MAX_CONTEXT} tokens."
            )

        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids

        return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids), 'labels': labels}

def collate_fn(
    batch,
    pad_token_id: int,
):
    max_len = max((len(item['input_ids']) for item in batch))

    input_ids = []
    attention_masks = []
    labels = []

    for item in batch:
        pad_length = max_len - len(item['input_ids'])

        input_ids.append(item['input_ids'] + [pad_token_id] * pad_length)

        attention_masks.append(item['attention_mask'] + [0] * pad_length)

        labels.append(item['labels'] + [-100] * pad_length)

    return {
        "input_ids": torch.tensor(
            input_ids,
            dtype=torch.long,
        ),
        "attention_mask": torch.tensor(
            attention_masks,
            dtype=torch.long,
        ),
        "labels": torch.tensor(
            labels,
            dtype=torch.long,
        ),
    }

# ADDED: Stage 1 datasets use thousands of external summary pairs.
phase1_train_dataset = NewsSimplificationDataset(phase1_train_df, tokenizer)

phase1_val_dataset = NewsSimplificationDataset(phase1_val_df, tokenizer)

# CHANGED: Stage 2 datasets use only the project Summary + Aspects data.
phase2_train_dataset = NewsSimplificationDataset(phase2_train_df, tokenizer)

phase2_val_dataset = NewsSimplificationDataset(phase2_val_df, tokenizer)

safe_collator = partial(collate_fn, pad_token_id=tokenizer.pad_token_id)

print('Stage 1 training examples:', len(phase1_train_dataset))
print('Stage 1 validation examples:', len(phase1_val_dataset))
print('Stage 2 training examples:', len(phase2_train_dataset))
print('Stage 2 validation examples:', len(phase2_val_dataset))

## Load model with 4-bit QLoRA

In [ ]:
# NOTE: QLoRA architecture is intentionally unchanged from the baseline.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=COMPUTE_DTYPE,
)

print("Model context limit:", model.config.max_position_embeddings)
if MAX_CONTEXT > model.config.max_position_embeddings:
    raise ValueError("MAX_CONTEXT exceeds the model context limit.")

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules="all-linear",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Fine-tune

In [ ]:
# ------------------------------------------------------------------
# ADDED: Optimization 3 - generate a small validation sample during
# training and select the best LoRA adapter by actual output quality.
#
# Stage 1 quality = equal mean of:
#   - Summary ROUGE-L
#   - SARI / 100
#
# Stage 2 quality = equal mean of:
#   - Summary ROUGE-L
#   - SARI / 100
#   - Aspect Macro ROUGE-L
#   - Structured-output completion
#
# Equal weighting avoids making any single aspect label dominate.
# ------------------------------------------------------------------
def generate_quality_output(
    model,
    article: str,
    prompt_builder,
):
    prompt, _, _ = fit_prompt_to_context(article, QUALITY_EVAL_MAX_NEW_TOKENS, prompt_builder=prompt_builder)

    device = next((parameter.device for parameter in model.parameters() if parameter.device.type != 'meta'))

    inputs = tokenizer(prompt, add_special_tokens=True, truncation=False, return_tensors='pt')

    inputs = {key: value.to(device) for key, value in inputs.items()}

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=QUALITY_EVAL_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][input_length:]

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def evaluate_model_quality(
    model,
    frame: pd.DataFrame,
    stage: str,
    max_samples: int = QUALITY_EVAL_SAMPLES,
):
    if stage not in {"summary", "full"}:
        raise ValueError(
            "stage must be 'summary' or 'full'."
        )

    prompt_builder = build_summary_prompt if stage == 'summary' else build_prompt

    sample = frame.head(min(max_samples, len(frame)))

    was_training = model.training
    old_use_cache = model.config.use_cache

    model.eval()
    model.config.use_cache = True

    rouge_l_scores = []
    sari_scores = []
    flesch_scores = []
    aspect_scores = []
    structure_scores = []

    for _, row in sample.iterrows():
        raw_output = generate_quality_output(model, row['article'], prompt_builder)

        generated_summary = quality_extract_summary(raw_output, stage)

        rouge_l_scores.append(quality_rouge_l(row['summary'], generated_summary))

        sari_scores.append(compute_sari_score(row['article'], generated_summary, row['summary']))

        flesch_scores.append(textstat.flesch_reading_ease(generated_summary) if generated_summary else 0.0)

        if stage == "full":
            generated_aspects = quality_extract_aspects(raw_output)

            aspect_scores.append(quality_aspect_macro_rouge_l(row['aspects'], generated_aspects))

            structure_scores.append(float(quality_has_complete_structure(raw_output)))

    model.config.use_cache = old_use_cache

    if was_training:
        model.train()

    mean_rouge_l = float(np.mean(rouge_l_scores))
    mean_sari = float(np.mean(sari_scores))
    mean_flesch = float(np.mean(flesch_scores))

    if stage == "summary":
        # ADDED: balanced summary checkpoint score.
        quality_score = float(np.mean([mean_rouge_l, mean_sari / 100.0]))

        return {'quality_score': quality_score, 'rougeL': mean_rouge_l, 'sari': mean_sari, 'flesch': mean_flesch}

    mean_aspect = float(np.mean(aspect_scores))
    mean_structure = float(np.mean(structure_scores))

    # ADDED: balanced final-task checkpoint score.
    quality_score = float(np.mean([mean_rouge_l, mean_sari / 100.0, mean_aspect, mean_structure]))

    return {
        "quality_score": quality_score,
        "rougeL": mean_rouge_l,
        "sari": mean_sari,
        "flesch": mean_flesch,
        "aspect_macro_rougeL": mean_aspect,
        "structure_completion": mean_structure,
    }

class QualityCheckpointCallback(TrainerCallback):
    """
    ADDED: selects/restores the adapter that produces the best validation
    outputs instead of trusting eval_loss alone.
    """

    def __init__(
        self,
        validation_frame,
        stage,
        best_dir,
        patience,
    ):
        self.validation_frame = validation_frame
        self.stage = stage
        self.best_dir = best_dir
        self.patience = patience

        self.best_score = -float("inf")
        self.best_step = None
        self.best_state_dict = None
        self.bad_evaluations = 0
        self.history = []

    def on_evaluate(
        self,
        args,
        state,
        control,
        model=None,
        metrics=None,
        **kwargs,
    ):
        quality = evaluate_model_quality(model, self.validation_frame, stage=self.stage)

        record = {'step': int(state.global_step), **quality}
        self.history.append(record)

        print('\nQuality-aware validation:', json.dumps(record, ensure_ascii=False, indent=2))

        score = quality["quality_score"]

        if score > self.best_score:
            self.best_score = score
            self.best_step = int(state.global_step)
            self.bad_evaluations = 0

            current_state = get_peft_model_state_dict(model)

            self.best_state_dict = {key: value.detach().cpu().clone() for key, value in current_state.items()}

            os.makedirs(self.best_dir, exist_ok=True)

            model.save_pretrained(self.best_dir)
            tokenizer.save_pretrained(self.best_dir)

            print('New best quality adapter saved:', self.best_dir)

        else:
            self.bad_evaluations += 1

            if (
                self.bad_evaluations
                >= self.patience
            ):
                print(f'Quality score has not improved for {self.patience} evaluations; stopping training.')
                control.should_training_stop = True

        return control

    def restore_best(self, model):
        if self.best_state_dict is None:
            raise RuntimeError(
                "No quality checkpoint was captured."
            )

        set_peft_model_state_dict(model, self.best_state_dict)

        print(f'Restored quality-selected adapter from step {self.best_step} (score={self.best_score:.4f}).')

# ------------------------------------------------------------------
# ADDED: Stage 1 - external summary-first curriculum fine-tuning.
# ------------------------------------------------------------------
phase1_quality_callback = QualityCheckpointCallback(
    validation_frame=phase1_val_df,
    stage="summary",
    best_dir=PHASE1_MODEL_DIR,
    patience=QUALITY_EARLY_STOPPING_PATIENCE,
)

phase1_training_args = TrainingArguments(
    output_dir=os.path.join(
        PHASE1_MODEL_DIR,
        "checkpoints",
    ),
    num_train_epochs=PHASE1_NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=PHASE1_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    max_grad_norm=MAX_GRAD_NORM,
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=PHASE1_EVAL_STEPS,
    save_strategy="steps",
    save_steps=PHASE1_SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=False,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to=[],
    seed=SEED,
    data_seed=SEED,
)

phase1_trainer = Trainer(
    model=model,
    args=phase1_training_args,
    train_dataset=phase1_train_dataset,
    eval_dataset=phase1_val_dataset,
    data_collator=safe_collator,
    callbacks=[
        phase1_quality_callback,
    ],
)

phase1_result = phase1_trainer.train()

print(phase1_result)

# ADDED: continue Stage 2 from the best generated-summary checkpoint,
# not simply from the final Stage-1 step.
phase1_quality_callback.restore_best(phase1_trainer.model)

phase1_quality_log_path = os.path.join(RESULTS_DIR, f'{RESULT_PREFIX}_phase1_quality_log.csv')

pd.DataFrame(phase1_quality_callback.history).to_csv(phase1_quality_log_path, index=False)

# ------------------------------------------------------------------
# ADDED: Stage 2 - project-specific Summary + 8 Aspects fine-tuning.
# The same LoRA adapter is continued; the base OPT is NOT reloaded.
# ------------------------------------------------------------------
model = phase1_trainer.model
model.config.use_cache = False

phase2_quality_callback = QualityCheckpointCallback(
    validation_frame=phase2_val_df,
    stage="full",
    best_dir=FINETUNED_MODEL_DIR,
    patience=QUALITY_EARLY_STOPPING_PATIENCE,
)

phase2_training_args = TrainingArguments(
    output_dir=os.path.join(
        FINETUNED_MODEL_DIR,
        "checkpoints",
    ),
    num_train_epochs=PHASE2_NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=PHASE2_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    max_grad_norm=MAX_GRAD_NORM,
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=PHASE2_EVAL_STEPS,
    save_strategy="steps",
    save_steps=PHASE2_SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=False,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to=[],
    seed=SEED,
    data_seed=SEED,
)

phase2_trainer = Trainer(
    model=model,
    args=phase2_training_args,
    train_dataset=phase2_train_dataset,
    eval_dataset=phase2_val_dataset,
    data_collator=safe_collator,
    callbacks=[
        phase2_quality_callback,
    ],
)

phase2_result = phase2_trainer.train()

print(phase2_result)

# ADDED: final model = best balanced validation-output checkpoint.
phase2_quality_callback.restore_best(phase2_trainer.model)

# Preserve the original downstream variable name and flow.
trainer = phase2_trainer

phase2_quality_log_path = os.path.join(RESULTS_DIR, f'{RESULT_PREFIX}_phase2_quality_log.csv')

pd.DataFrame(phase2_quality_callback.history).to_csv(phase2_quality_log_path, index=False)

print('Stage 1 best quality step:', phase1_quality_callback.best_step)
print('Stage 1 best quality score:', phase1_quality_callback.best_score)
print('Stage 2 best quality step:', phase2_quality_callback.best_step)
print('Stage 2 best quality score:', phase2_quality_callback.best_score)

## Save model and training log

In [ ]:
# CHANGED: trainer now points to the quality-selected Stage-2 model.
trainer.save_model(FINETUNED_MODEL_DIR)
tokenizer.save_pretrained(FINETUNED_MODEL_DIR)

# ADDED: keep separate logs for both curriculum stages.
phase1_training_log_path = os.path.join(RESULTS_DIR, f'{RESULT_PREFIX}_phase1_training_log.csv')

phase2_training_log_path = os.path.join(RESULTS_DIR, f'{RESULT_PREFIX}_phase2_training_log.csv')

pd.DataFrame(phase1_trainer.state.log_history).to_csv(phase1_training_log_path, index=False)

pd.DataFrame(phase2_trainer.state.log_history).to_csv(phase2_training_log_path, index=False)

metadata = {
    "base_model": MODEL_NAME,
    "saved_model_dir": FINETUNED_MODEL_DIR,
    "training_method": (
        "4-bit QLoRA / PEFT with external-summary curriculum"
    ),
    # ADDED: record all optimization stages for reproducibility.
    "optimization_flow": [
        "Stage 1: external news Article -> Summary",
        "Stronger summary prompt",
        "Quality-aware checkpoint evaluation",
        "Source TF-IDF keyword guidance",
        "Stage 2: project Article -> Summary + 8 Aspects",
    ],
    # CHANGED: Stage-1 data provenance now points only to local prepared files.
    "stage1_summary_data": {
        "train_path": STAGE1_TRAIN_PATH,
        "validation_path": STAGE1_VAL_PATH,
        "train_records_used": len(
            phase1_train_df
        ),
        "validation_records_used": len(
            phase1_val_df
        ),
        "download_logic_in_training_notebook": False,
    },
    "project_data": {
        "train_records_used": len(
            phase2_train_df
        ),
        "validation_records_used": len(
            phase2_val_df
        ),
        "test_records_held_out": len(
            test_df
        ),
    },
    "max_context": MAX_CONTEXT,
    "max_target_tokens": MAX_TARGET_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "stage1": {
        "max_epochs": PHASE1_NUM_TRAIN_EPOCHS,
        "learning_rate": PHASE1_LEARNING_RATE,
        "best_quality_step": (
            phase1_quality_callback.best_step
        ),
        "best_quality_score": (
            phase1_quality_callback.best_score
        ),
        "articles_truncated": int(
            phase1_train_df[
                "article_was_truncated"
            ].sum()
        ),
    },
    "stage2": {
        "max_epochs": PHASE2_NUM_TRAIN_EPOCHS,
        "learning_rate": PHASE2_LEARNING_RATE,
        "best_quality_step": (
            phase2_quality_callback.best_step
        ),
        "best_quality_score": (
            phase2_quality_callback.best_score
        ),
        "articles_truncated": int(
            phase2_train_df[
                "article_was_truncated"
            ].sum()
        ),
    },
    "quality_checkpoint_selection": {
        "validation_generation_samples": (
            QUALITY_EVAL_SAMPLES
        ),
        "patience": (
            QUALITY_EARLY_STOPPING_PATIENCE
        ),
        "stage1_metrics": [
            "ROUGE-L",
            "SARI",
        ],
        "stage2_metrics": [
            "ROUGE-L",
            "SARI",
            "Aspect Macro ROUGE-L",
            "Structured Output Completion",
        ],
    },
    "keyword_guidance": {
        "enabled": USE_KEYWORD_GUIDANCE,
        "method": "source-only TF-IDF",
        "top_k": KEYWORD_TOP_K,
        "fit_on": (
            "external training articles + "
            "project training articles only"
        ),
    },
    "seed": SEED,
    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": "all-linear",
    },
    "training": {
        "batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": (
            GRADIENT_ACCUMULATION_STEPS
        ),
        "effective_batch_size": (
            TRAIN_BATCH_SIZE
            * GRADIENT_ACCUMULATION_STEPS
        ),
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "max_grad_norm": MAX_GRAD_NORM,
        "optimizer": "paged_adamw_8bit",
        "lr_scheduler": "cosine",
        "compute_dtype": str(COMPUTE_DTYPE),
    },
    "software": {
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "accelerate": accelerate.__version__,
        "bitsandbytes": bnb.__version__,
    },
}

with open(
    os.path.join(
        FINETUNED_MODEL_DIR,
        "training_metadata.json",
    ),
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Saved final PEFT adapter and tokenizer to '{FINETUNED_MODEL_DIR}'")
print(f"Saved Stage-1 training log to '{phase1_training_log_path}'")
print(f"Saved Stage-2 training log to '{phase2_training_log_path}'")
print(f"Saved Stage-1 quality log to '{phase1_quality_log_path}'")
print(f"Saved Stage-2 quality log to '{phase2_quality_log_path}'")

## Single-pass test generation

In [ ]:
finetuned_model = trainer.model
finetuned_model.eval()
finetuned_model.config.use_cache = True

if hasattr(
    finetuned_model,
    "gradient_checkpointing_disable",
):
    finetuned_model.gradient_checkpointing_disable()

torch.cuda.empty_cache()

OUTPUT_LABELS = ('Event', 'People', 'Organization', 'Date & Time', 'Location', 'Causes', 'Effects', 'Countermeasures')

ASPECT_FIELD_MAP = {
    "Event": "event",
    "People": "people",
    "Organization": "organization",
    "Date & Time": "date_time",
    "Location": "location",
    "Causes": "causes",
    "Effects": "effects",
    "Countermeasures": "countermeasures",
}

# ADDED: fail immediately if labels ever become inconsistent.
assert OUTPUT_LABELS == tuple(
    ASPECT_FIELD_MAP.keys()
), (
    "OUTPUT_LABELS and ASPECT_FIELD_MAP do not match: "
    f"{OUTPUT_LABELS} vs "
    f"{tuple(ASPECT_FIELD_MAP.keys())}"
)

GENERATION_DEVICE = next(
    parameter.device
    for parameter in finetuned_model.parameters()
    if parameter.device.type != "meta"
)

print('Generation device:', GENERATION_DEVICE)

def generate_article_output(article: str):
    # CHANGED: build_prompt now automatically adds stronger summary
    # guidance and source-only keywords while preserving all labels.
    (
        prompt,
        input_token_length,
        article_was_truncated,
    ) = fit_prompt_to_context(
        article,
        MAX_NEW_TOKENS,
        prompt_builder=build_prompt,
    )

    inputs = tokenizer(prompt, add_special_tokens=True, truncation=False, return_tensors='pt')

    inputs = {key: value.to(GENERATION_DEVICE) for key, value in inputs.items()}

    input_length = inputs['input_ids'].shape[1]

    start_time = time.perf_counter()

    with torch.inference_mode():
        output_ids = finetuned_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.perf_counter() - start_time

    generated_ids = output_ids[0][input_length:]

    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return {
        "raw_output": raw_output,
        "generation_time_sec": elapsed,
        "input_token_length": (
            input_token_length
        ),
        "article_was_truncated": (
            article_was_truncated
        ),
        # ADDED: save the deterministic source hints used for this sample.
        "source_keywords": "; ".join(
            extract_source_keywords(
                article
            )
        ),
    }

def extract_summary(raw_text: str) -> str:
    match = re.search('(?is)^\\s*Summary\\s*:\\s*(.*?)(?=\\n\\s*Aspects\\s*:|\\Z)', str(raw_text))

    return re.sub('\\s+', ' ', match.group(1)).strip() if match else ''

def extract_aspects(raw_text: str) -> dict:
    text = str(raw_text)
    extracted = {}

    # CHANGED: iterate over the mapping directly to prevent
    # the previous CausesEffects KeyError class of problem.
    for label, field_name in ASPECT_FIELD_MAP.items():
        match = re.search(f'(?im)^\\s*{re.escape(label)}\\s*:\\s*(.*?)\\s*$', text)

        extracted[field_name] = match.group(1).strip() if match else ''

    return extracted

def has_complete_structure(
    raw_text: str,
) -> bool:
    text = str(raw_text)

    ordered_labels = ['Summary', 'Aspects', *OUTPUT_LABELS]

    positions = []

    for label in ordered_labels:
        found = list(re.finditer(f'(?im)^\\s*{re.escape(label)}\\s*:', text))

        if len(found) != 1:
            return False

        positions.append(found[0].start())

    return positions == sorted(positions)

def exact_label_completion_rate(
    raw_text: str,
) -> float:
    text = str(raw_text)

    labels = ['Summary', 'Aspects', *OUTPUT_LABELS]

    present = sum((bool(re.search(f'(?im)^\\s*{re.escape(label)}\\s*:', text)) for label in labels))

    return present / len(labels)

In [ ]:
generated_rows = []

for position, (_, row) in enumerate(test_df.iterrows(), start=1):
    generated_rows.append(generate_article_output(row["article"]))

    if position % 10 == 0 or position == len(test_df):
        print(f"Generated {position}/{len(test_df)} responses")

generated_df = pd.DataFrame(generated_rows)
test_df = pd.concat([test_df.reset_index(drop=True), generated_df], axis=1)

test_df["generated_summary"] = test_df["raw_output"].apply(extract_summary)
test_df["generated_aspects"] = test_df["raw_output"].apply(extract_aspects)
test_df["structured_output_complete"] = test_df["raw_output"].apply(has_complete_structure)
test_df["exact_label_completion_rate"] = test_df["raw_output"].apply(exact_label_completion_rate)

test_df[
    [
        "summary",
        "generated_summary",
        "raw_output",
        "structured_output_complete",
        "exact_label_completion_rate",
    ]
].head()

## Evaluate

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_rouge(
    reference: str,
    hypothesis: str,
):
    scores = scorer.score(str(reference), str(hypothesis))

    return {'rouge1': scores['rouge1'].fmeasure, 'rouge2': scores['rouge2'].fmeasure, 'rougeL': scores['rougeL'].fmeasure}

def rouge_l(
    reference: str,
    hypothesis: str,
) -> float:
    return scorer.score(str(reference), str(hypothesis))['rougeL'].fmeasure

def word_count(text: str) -> int:
    return max(1, len(str(text).split()))

def aspect_macro_rouge_l(
    reference_aspects: dict,
    generated_aspects: dict,
) -> float:
    return float(np.mean([rouge_l(reference_aspects[key], generated_aspects.get(key, '')) for key in ASPECT_KEYS]))

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []
sari_scores = []                 # ADDED: simplification-specific metric.
full_target_rougeL_scores = []
aspect_macro_scores = []
flesch_scores = []
compression_ratios = []

for _, row in test_df.iterrows():
    summary_rouge = compute_rouge(row['summary'], row['generated_summary'])

    rouge1_scores.append(summary_rouge['rouge1'])
    rouge2_scores.append(summary_rouge['rouge2'])
    rougeL_scores.append(summary_rouge['rougeL'])

    # ADDED: evaluate KEEP / DELETE / ADD simplification behaviour.
    sari_scores.append(compute_sari_score(row['article'], row['generated_summary'], row['summary']))

    full_target_rougeL_scores.append(rouge_l(row['training_target'], row['raw_output']))

    aspect_macro_scores.append(aspect_macro_rouge_l(row['aspects'], row['generated_aspects']))

    flesch_scores.append(textstat.flesch_reading_ease(row['generated_summary']) if row['generated_summary'] else 0.0)

    compression_ratios.append(word_count(row['generated_summary']) / word_count(row['article']))

test_df["rouge1"] = rouge1_scores
test_df["rouge2"] = rouge2_scores
test_df["rougeL"] = rougeL_scores
test_df["sari"] = sari_scores                  # ADDED
test_df['full_target_rougeL'] = full_target_rougeL_scores
test_df['aspect_macro_rougeL'] = aspect_macro_scores
test_df['flesch_reading_ease'] = flesch_scores
test_df['compression_ratio'] = compression_ratios

## Save evaluation results

In [ ]:
summary_metrics = pd.DataFrame(
    {
        "metric": [
            "ROUGE-1",
            "ROUGE-2",
            "ROUGE-L",
            "SARI",                         # ADDED
            "Full Target ROUGE-L",
            "Aspect Macro ROUGE-L",
            "Flesch Reading Ease",
            "Compression Ratio",
            "Generation Time (sec)",
            "Structured Output Completion Rate",
            "Exact Label Completion Rate",
            "Test Article Truncation Rate",
        ],
        "mean": [
            test_df["rouge1"].mean(),
            test_df["rouge2"].mean(),
            test_df["rougeL"].mean(),
            test_df["sari"].mean(),         # ADDED
            test_df["full_target_rougeL"].mean(),
            test_df["aspect_macro_rougeL"].mean(),
            test_df["flesch_reading_ease"].mean(),
            test_df["compression_ratio"].mean(),
            test_df["generation_time_sec"].mean(),
            test_df["structured_output_complete"].mean(),
            test_df["exact_label_completion_rate"].mean(),
            test_df["article_was_truncated"].mean(),
        ],
        "std": [
            test_df["rouge1"].std(),
            test_df["rouge2"].std(),
            test_df["rougeL"].std(),
            test_df["sari"].std(),          # ADDED
            test_df["full_target_rougeL"].std(),
            test_df["aspect_macro_rougeL"].std(),
            test_df["flesch_reading_ease"].std(),
            test_df["compression_ratio"].std(),
            test_df["generation_time_sec"].std(),
            test_df["structured_output_complete"].std(),
            test_df["exact_label_completion_rate"].std(),
            test_df["article_was_truncated"].std(),
        ],
    }
)

print(summary_metrics.to_string(index=False))

per_example_path = os.path.join(RESULTS_DIR, f'{RESULT_PREFIX}_test_results.csv')

summary_path = os.path.join(RESULTS_DIR, f'{RESULT_PREFIX}_evaluation_summary.csv')

save_df = test_df.copy()

save_df['aspects'] = save_df['aspects'].apply(lambda value: json.dumps(value, ensure_ascii=False))

save_df['generated_aspects'] = save_df['generated_aspects'].apply(lambda value: json.dumps(value, ensure_ascii=False))

save_df.drop(columns=['training_target'], errors='ignore').to_csv(per_example_path, index=False)

summary_metrics.to_csv(summary_path, index=False)

print(f"Saved test results to '{per_example_path}'")
print(f"Saved evaluation summary to '{summary_path}'")